# Module 02: Khí động lực học / Aerodynamics
**AE2.008 · USTH: Aerodynamics and Flight Mechanics**

🇻🇳 Giải lại các bài **Anderson 1.5, 3.3, 3.4, 3.6, 3.7, 3.18, 4.2, 4.5, 4.10, 4.11, 4.12**, vẽ đường cong C_L theo góc tấn của NACA 2412 (số liệu từ bài 1.6) và đường cực lý thuyết, rồi `assert` các đáp án quiz.

🇬🇧 Re-solves **Anderson 1.5, 3.3, 3.4, 3.6, 3.7, 3.18, 4.2, 4.5, 4.10, 4.11, 4.12**, plots the NACA 2412 C_L-versus-angle curve (data from Problem 1.6) and a theoretical polar, then asserts the quiz answers.

In [ ]:
import math
G0, R_AIR, GAMMA = 9.80665, 287.05287, 1.4
P0, T0, RHO0 = 101325.0, 288.15, 1.225
FT, PSI = 0.3048, 6894.757

def isa(h_m):
    """ISA 0-20 km: (T [K], p [Pa], rho [kg/m3]). Troposphere lapse -6.5 K/km, isothermal 216.65 K above 11 km."""
    if h_m <= 11000:
        L = -0.0065; T = T0 + L*h_m; p = P0*(T/T0)**(-G0/(L*R_AIR))
    else:
        T = 216.65; p11 = P0*(T/T0)**(-G0/(-0.0065*R_AIR)); p = p11*math.exp(-G0*(h_m-11000)/(R_AIR*T))
    return T, p, p/(R_AIR*T)

def isa_ft(h_ft): return isa(h_ft*FT)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


## 1. Hệ số lực / Force coefficients (Anderson 1.5)

In [ ]:
a = math.radians(12); cn, ca = 1.2, 0.03
cl = cn*math.cos(a) - ca*math.sin(a); cd = cn*math.sin(a) + ca*math.cos(a)
print(f"1.5: cl = {cl:.4f}, cd = {cd:.4f}"); assert abs(cl-1.1675) < 5e-4 and abs(cd-0.2788) < 5e-4

## 2. Bernoulli: Venturi, hầm gió, Pitot (Anderson 3.3, 3.4, 3.6, 3.7)

In [ ]:
rho = 1.23
vac = 0.5*rho*90**2*(1/0.85**2 - 1);                 print("3.3 vacuum =", round(vac), "N/m2"); assert abs(vac-1913) < 2
dp = 1.36e4*9.81*0.10; V = math.sqrt(2*dp/(rho*(1-1/144))); print("3.4 V =", round(V, 1), "m/s"); assert abs(V-147.8) < 0.1
V6 = math.sqrt(2*(1.07e5-1.01e5)/rho);               print("3.6 V =", round(V6, 1), "m/s"); assert abs(V6-98.8) < 0.1
cp = 1 - (130/V6)**2;                                print("3.7 Cp =", round(cp, 3));     assert abs(cp+0.732) < 1e-3
G = 6/(rho*30);                                       print("3.18 Gamma =", round(G, 4), "m2/s"); assert abs(G-0.1626) < 1e-3

## 3. Biên dạng NACA 2412 (Anderson 1.6, 4.1, 4.2, 4.10, 4.11, 4.12)

In [ ]:
# Table from Problem 1.6
alpha = np.array([-2, 0, 2, 4, 6, 8, 10, 12, 14.0])
cl_t  = np.array([0.05, 0.25, 0.44, 0.64, 0.85, 1.08, 1.26, 1.43, 1.56])
cd_t  = np.array([0.006, 0.006, 0.006, 0.007, 0.0075, 0.0092, 0.0115, 0.0150, 0.0186])
cm_t  = np.array([-0.042, -0.040, -0.038, -0.036, -0.036, -0.036, -0.034, -0.030, -0.025])
slope = np.polyfit(alpha[:6], cl_t[:6], 1); print("lift slope = %.4f /deg (2*pi/rad = %.4f /deg)" % (slope[0], 2*math.pi*math.pi/180))
# 4.2
c = 2.0; q = 0.5*1.23*50**2; cl42 = 1353/(q*c); print("4.2 cl =", round(cl42, 3)); assert abs(cl42-0.44) < 1e-9
assert alpha[np.argmin(abs(cl_t-cl42))] == 2          # alpha = 2 deg
# 4.1 (cl and cm at 4 deg from Problem 4.10)
q1 = 0.5*0.002377*50**2; L1 = q1*2*0.64; M1 = q1*4*(-0.036); print(f"4.1 L' = {L1:.3f} lb/ft, M'c/4 = {M1:.4f} ft lb/ft"); assert abs(L1-3.80) < 0.01 and abs(M1+0.428) < 0.002   # c_l=0.64, c_m=-0.036 at 4 deg (table of Problem 1.6); matches the Solution Manual 3.80 / -0.428
# 4.5 thin symmetric airfoil
cl45 = 2*math.pi*math.radians(1.5); cm45 = -cl45/4; print(f"4.5 cl = {cl45:.4f}, cm,LE = {cm45:.4f}"); assert abs(cl45-0.1645) < 1e-4
# 4.10 aerodynamic centre
hac = 0.25 - (-0.037-(-0.045))/(0.65-(-0.39)); print("4.10 x_ac/c =", round(hac, 4)); assert abs(hac-0.2423) < 1e-4
# 4.11, 4.12
r3 = isa(3000)[2]; q3 = 0.5*r3*60**2; Lp = q3*2*0.65; Gam = Lp/(r3*60)
print(f"4.11 rho = {r3:.4f}, L' = {Lp:.0f} N/m; 4.12 Gamma = {Gam:.1f} m2/s"); assert abs(Lp-2127) < 2 and abs(Gam-39.0) < 0.1

## 4. Đồ thị / Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(alpha, cl_t, "o-"); ax[0].set(xlabel="alpha [deg]", ylabel="c_l", title="NACA 2412: c_l vs alpha"); ax[0].grid(True)
ax[1].plot(cd_t, cl_t, "s-"); ax[1].set(xlabel="c_d", ylabel="c_l", title="NACA 2412: drag polar (section)"); ax[1].grid(True)
fig.tight_layout(); fig.savefig("naca2412_curves.png", dpi=100); print("saved naca2412_curves.png")
best = np.argmax(cl_t/cd_t); print(f"max section c_l/c_d = {cl_t[best]/cd_t[best]:.0f} at alpha = {alpha[best]} deg")

## 5. Kiểm quiz / Quiz self-check

In [ ]:
AR = 12**2/20; assert AR == 7.2                         # generated question
assert abs((1-(math.sqrt(2))**2)+1) < 1e-12              # Cp = -1 at V = sqrt(2) V_inf
assert abs(4 - (200/100)**2) < 1e-12                     # lift x4 when speed x2
print("All Module 02 checks passed.")

## 6. Câu bổ sung / Extra questions

In [ ]:
q100 = 0.5*1.225*100**2; assert abs(q100-6125) < 1e-9
L1 = 0.5*1.225*60**2*20*0.5; assert abs(L1-22050) < 1e-6
cdi = 0.8**2/(math.pi*0.9*8); assert abs(cdi-0.02829) < 1e-4
Re = 1.225*50*2/1.789e-5; assert abs(Re/1e6-6.85) < 0.01
assert abs(isa(3000)[2]/1.225-0.742) < 0.002
print("Extra Module 02 checks passed.")